# Sales CSV - Metadata documentation

This notebook adds Unity Catalog descriptions to the completed CSV pipeline without touching table data. Keeping metadata work separate makes documentation repeatable and lets catalog users understand source lineage, quality rules, and analytical grains directly where the objects are discovered.

In [0]:
dbutils.widgets.removeAll()

## Documentation scope and targets

The environment widget resolves all Bronze, Silver, rejected, and Gold tables. The notebook documents both business columns and technical lineage fields so operational and analytical users share the same definitions.

In [0]:
dbutils.widgets.text(
    "environment",
    "dev",
    "Environment"
)

environment = dbutils.widgets.get("environment").lower()

if environment not in ["dev", "prod"]:
    raise ValueError(
        "Environment must be either 'dev' or 'prod'."
    )

config = {
    "dev": {
        "catalog": "salescsv_dev"
    },
    "prod": {
        "catalog": "salescsv_prod"
    }
}

env = config[environment]

catalog = env["catalog"]

# Bronze
product_bronze_table = (
    f"{catalog}.bronze.product_catalog_raw"
)

inventory_bronze_table = (
    f"{catalog}.bronze.inventory_transactions_raw"
)

# Silver
silver_table = (
    f"{catalog}.silver.inventory_movements"
)

rejected_table = (
    f"{catalog}.silver.rejected_transactions"
)

# Gold
product_gold_table = (
    f"{catalog}.gold.inventory_by_product"
)

warehouse_gold_table = (
    f"{catalog}.gold.inventory_by_warehouse"
)

low_stock_gold_table = (
    f"{catalog}.gold.low_stock_products"
)

print("=" * 60)
print("SALES CSV - METADATA DOCUMENTATION")
print("=" * 60)
print(f"Environment       : {environment}")
print(f"Catalog           : {catalog}")
print("=" * 60)

## Table-level contracts

Table comments summarize each layer's purpose: source-shaped CSV data in Bronze, standardized and quality-routed movements in Silver, and inventory aggregates or exception views in Gold.

In [0]:


table_comments = {
    product_bronze_table:
        "Raw product catalog ingested from CSV files in Azure Data Lake Storage. "
        "The table preserves source attributes and technical ingestion metadata "
        "for traceability and downstream processing.",

    inventory_bronze_table:
        "Raw inventory transactions ingested from CSV files in Azure Data Lake Storage. "
        "The table preserves source values and technical ingestion metadata before "
        "Silver data-quality and business transformations.",

    silver_table:
        "Validated and enriched inventory movements produced by joining inventory "
        "transactions with the product catalog. Contains standardized data types, "
        "product attributes, inventory impact calculations, and only records that "
        "passed defined data-quality rules.",

    rejected_table:
        "Inventory transactions rejected during Silver processing because they failed "
        "one or more business data-quality rules. Records are retained for auditing, "
        "traceability, and troubleshooting.",

    product_gold_table:
        "Current inventory position aggregated by product. Includes receipts, sales, "
        "returns, adjustments, current stock, inventory value, reorder thresholds, "
        "and operational stock status for analytics and Databricks Genie.",

    warehouse_gold_table:
        "Inventory summary aggregated by warehouse, including product counts, transaction "
        "volume, current inventory units, inventory value, and inventory movement statistics.",

    low_stock_gold_table:
        "Products whose calculated current stock is at or below the configured reorder "
        "level. Designed for operational low-stock monitoring, dashboards, and alerts."
}

for table_name, comment in table_comments.items():
    spark.sql(f"""
        COMMENT ON TABLE {table_name}
        IS '{comment}'
    """)

print("Table comments applied successfully.")

## Safe column-comment helper

Column comments are applied only when a column exists in the current table schema. This allows the documentation notebook to be rerun safely across compatible deployments while reporting any documented field that is not yet available.

In [0]:

def apply_column_comments(table_name, comments):
    existing_columns = {
        field.name
        for field in spark.table(table_name).schema.fields
    }

    for column_name, comment in comments.items():

        if column_name in existing_columns:

            spark.sql(f"""
                ALTER TABLE {table_name}
                ALTER COLUMN `{column_name}`
                COMMENT '{comment}'
            """)

    print(
        f"Column comments applied: {table_name}"
    )

In [0]:
product_bronze_comments = {
    "product_id":
        "Unique source identifier of the product.",

    "product_name":
        "Product name provided by the source CSV dataset.",

    "category":
        "Business category assigned to the product.",

    "supplier":
        "Supplier associated with the product.",

    "unit_cost":
        "Source unit cost of the product.",

    "reorder_level":
        "Inventory threshold used to identify products requiring replenishment.",

    "source_file":
        "Name of the CSV source file from which the record was ingested.",

    "source_file_path":
        "Full Azure Data Lake Storage path of the source CSV file.",

    "ingestion_timestamp":
        "Timestamp associated with the Bronze ingestion execution."
}

apply_column_comments(
    product_bronze_table,
    product_bronze_comments
)

inventory_bronze_comments = {
    "transaction_id":
        "Unique source identifier of the inventory transaction.",

    "transaction_timestamp":
        "Timestamp provided by the source system for the inventory transaction.",

    "product_id":
        "Identifier of the product affected by the inventory transaction.",

    "warehouse":
        "Warehouse where the inventory transaction occurred.",

    "transaction_type":
        "Inventory movement type such as RECEIPT, SALE, RETURN, or ADJUSTMENT.",

    "quantity":
        "Source quantity associated with the inventory transaction.",

    "source_file":
        "Name of the CSV source file from which the transaction was ingested.",

    "source_file_path":
        "Full Azure Data Lake Storage path of the source CSV file.",

    "ingestion_timestamp":
        "Timestamp associated with the Bronze ingestion execution."
}

apply_column_comments(
    inventory_bronze_table,
    inventory_bronze_comments
)

In [0]:
silver_comments = {
    "transaction_id":
        "Unique identifier of the validated inventory transaction.",

    "transaction_timestamp":
        "Validated timestamp of the inventory transaction.",

    "product_id":
        "Product identifier used to join inventory transactions with the product catalog.",

    "product_name":
        "Product name enriched from the product catalog.",

    "category":
        "Product category enriched from the product catalog.",

    "supplier":
        "Product supplier enriched from the product catalog.",

    "warehouse":
        "Warehouse where the inventory movement occurred.",

    "transaction_type":
        "Validated inventory movement type.",

    "quantity":
        "Validated numeric quantity associated with the transaction.",

    "quantity_raw":
        "Original source quantity preserved before numeric conversion.",

    "unit_cost":
        "Product unit cost enriched from the product catalog.",

    "reorder_level":
        "Inventory threshold used to identify low-stock products.",

    "inventory_change":
        "Signed inventory impact of the transaction. Sales reduce inventory while receipts and returns increase inventory.",

    "inventory_value_change":
        "Monetary inventory impact calculated from inventory change and unit cost.",

    "ingestion_timestamp":
        "Timestamp associated with the Bronze ingestion execution.",

    "silver_processing_timestamp":
        "Timestamp associated with the Silver transformation execution."
}

apply_column_comments(
    silver_table,
    silver_comments
)

rejected_comments = {
    "transaction_id":
        "Unique identifier of the rejected inventory transaction.",

    "quantity_raw":
        "Original quantity value preserved for troubleshooting malformed source records.",

    "rejection_reason":
        "Business data-quality rule that caused the transaction to be rejected from the validated Silver dataset.",

    "source_file":
        "CSV source file containing the rejected transaction.",

    "ingestion_timestamp":
        "Timestamp associated with the Bronze ingestion execution."
}

apply_column_comments(
    rejected_table,
    rejected_comments
)

In [0]:
product_gold_comments = {
    "product_id":
        "Unique identifier of the product.",

    "product_name":
        "Business name of the product.",

    "category":
        "Business category assigned to the product.",

    "supplier":
        "Supplier associated with the product.",

    "unit_cost":
        "Current product unit cost used for inventory valuation.",

    "reorder_level":
        "Minimum inventory threshold used for replenishment decisions.",

    "total_received":
        "Total units received into inventory.",

    "total_sold":
        "Total units removed from inventory through sales.",

    "total_returned":
        "Total units returned back into inventory.",

    "total_adjustment":
        "Net inventory quantity resulting from manual adjustments.",

    "current_stock":
        "Calculated current inventory quantity based on all validated inventory movements.",

    "warehouse_count":
        "Number of warehouses containing inventory activity for the product.",

    "inventory_value":
        "Calculated monetary value of current inventory based on current stock and unit cost.",

    "stock_status":
        "Operational inventory status classified as IN_STOCK, LOW_STOCK, or OUT_OF_STOCK.",

    "gold_processing_timestamp":
        "Timestamp associated with the Gold aggregation execution."
}

apply_column_comments(
    product_gold_table,
    product_gold_comments
)

warehouse_gold_comments = {
    "warehouse":
        "Warehouse represented by the inventory summary.",

    "total_products":
        "Number of distinct products with inventory activity in the warehouse.",

    "total_transactions":
        "Number of distinct validated inventory transactions processed for the warehouse.",

    "total_units":
        "Net inventory units calculated from all inventory movements in the warehouse.",

    "inventory_value":
        "Net monetary inventory value calculated from inventory movements in the warehouse.",

    "average_inventory_change":
        "Average signed inventory quantity change per transaction.",

    "min_inventory_change":
        "Lowest signed inventory change observed for a transaction.",

    "max_inventory_change":
        "Highest signed inventory change observed for a transaction.",

    "gold_processing_timestamp":
        "Timestamp associated with the Gold aggregation execution."
}

apply_column_comments(
    warehouse_gold_table,
    warehouse_gold_comments
)

low_stock_comments = {
    "product_id":
        "Unique identifier of the product requiring inventory attention.",

    "product_name":
        "Business name of the low-stock product.",

    "category":
        "Business category of the low-stock product.",

    "supplier":
        "Supplier associated with the low-stock product.",

    "unit_cost":
        "Unit cost used to calculate current inventory value.",

    "reorder_level":
        "Configured stock threshold indicating when replenishment should be considered.",

    "current_stock":
        "Current calculated inventory quantity for the product.",

    "inventory_value":
        "Current monetary inventory value for the product.",

    "stock_status":
        "Current operational status of the product inventory.",

    "gold_processing_timestamp":
        "Timestamp associated with the Gold aggregation execution."
}

apply_column_comments(
    low_stock_gold_table,
    low_stock_comments
)